In [1]:
import os, glob, argparse, random
import numpy as np
import cv2
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


In [2]:

IMG_SIZE = 512                      # 학습/추론 해상도 (정사각 리사이즈)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ────────────────────────────────────────────────────────────── DICOM 로더
def load_dicom(path, size=IMG_SIZE):
    """DICOM → float32 [0,1] 그레이스케일 (뼈가 밝음). 노트북 load_xray와 같은 규약."""
    ds = pydicom.dcmread(path)
    a = ds.pixel_array.astype(np.float32)
    lo, hi = np.percentile(a, [0.5, 99.5])          # 로버스트 윈도: 핫/데드 픽셀 무시
    a = np.clip((a - lo) / (hi - lo + 1e-9), 0, 1)
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        a = 1.0 - a                                 # MONOCHROME1은 반전 → 뼈를 밝게
    a = cv2.resize(a, (size, size), interpolation=cv2.INTER_LINEAR)
    return a.astype(np.float32)


def load_mask(path, size=IMG_SIZE):
    """마스크 PNG → {0,1} float32. (채널 합 > 10)을 전경으로 (노트북 규약과 동일)."""
    m = cv2.imread(path)
    if m is None:
        return None
    fg = (m.sum(axis=2) > 10).astype(np.float32)
    fg = cv2.resize(fg, (size, size), interpolation=cv2.INTER_NEAREST)
    return fg.astype(np.float32)


# ────────────────────────────────────────────────────────────── Dataset
class DicomSegDataset(Dataset):
    """DICOM ↔ 마스크 PNG 쌍. stem(확장자 제외 파일명)으로 매칭."""

    def __init__(self, img_dir, mask_dir, augment=False):
        self.pairs = []
        for dp in sorted(glob.glob(os.path.join(img_dir, "*.dcm"))):
            stem = os.path.splitext(os.path.basename(dp))[0]
            mp = os.path.join(mask_dir, stem + ".png")
            if os.path.exists(mp):
                self.pairs.append((dp, mp))
        self.augment = augment
        if not self.pairs:
            raise RuntimeError(f"짝이 맞는 DICOM/마스크가 없음: {img_dir} / {mask_dir}")

    def __len__(self):
        return len(self.pairs)

    def _aug(self, img, msk):
        # 좌우 반전
        if random.random() < 0.5:
            img, msk = img[:, ::-1].copy(), msk[:, ::-1].copy()
        # 소각 회전 + 이동 (재현성보다 다양성; 42장은 너무 적어 증강이 중요)
        if random.random() < 0.7:
            H, W = img.shape
            ang = random.uniform(-8, 8)
            tx, ty = random.uniform(-0.05, 0.05) * W, random.uniform(-0.05, 0.05) * H
            M = cv2.getRotationMatrix2D((W / 2, H / 2), ang, 1.0)
            M[0, 2] += tx; M[1, 2] += ty
            img = cv2.warpAffine(img, M, (W, H), flags=cv2.INTER_LINEAR)
            msk = cv2.warpAffine(msk, M, (W, H), flags=cv2.INTER_NEAREST)
        # 밝기/대비 지터 (X선 노출 변동 흉내)
        if random.random() < 0.5:
            img = np.clip(img * random.uniform(0.85, 1.15) + random.uniform(-0.05, 0.05), 0, 1)
        return img, msk

    def __getitem__(self, i):
        dp, mp = self.pairs[i]
        img = load_dicom(dp)
        msk = load_mask(mp)
        if self.augment:
            img, msk = self._aug(img, msk)
        img = torch.from_numpy(img)[None]           # (1, H, W)
        msk = torch.from_numpy(msk)[None]           # (1, H, W)
        return img, msk


# ────────────────────────────────────────────────────────────── U-Net
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    """표준 U-Net. in_ch=1(그레이스케일), out_ch=1(L4 이진)."""

    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        self.d1 = DoubleConv(in_ch, base)
        self.d2 = DoubleConv(base, base * 2)
        self.d3 = DoubleConv(base * 2, base * 4)
        self.d4 = DoubleConv(base * 4, base * 8)
        self.bott = DoubleConv(base * 8, base * 16)
        self.pool = nn.MaxPool2d(2)
        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.u4 = DoubleConv(base * 16, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.u3 = DoubleConv(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.u2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.u1 = DoubleConv(base * 2, base)
        self.head = nn.Conv2d(base, out_ch, 1)      # 로짓(logits) 출력, sigmoid는 손실/추론에서

    def forward(self, x):
        c1 = self.d1(x)
        c2 = self.d2(self.pool(c1))
        c3 = self.d3(self.pool(c2))
        c4 = self.d4(self.pool(c3))
        b = self.bott(self.pool(c4))
        x = self.u4(torch.cat([self.up4(b), c4], 1))
        x = self.u3(torch.cat([self.up3(x), c3], 1))
        x = self.u2(torch.cat([self.up2(x), c2], 1))
        x = self.u1(torch.cat([self.up1(x), c1], 1))
        return self.head(x)


# ────────────────────────────────────────────────────────────── 손실 / 지표
def dice_loss(logits, target, eps=1e-6):
    prob = torch.sigmoid(logits)
    num = 2 * (prob * target).sum(dim=(2, 3))
    den = prob.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + eps
    return (1 - (num + eps) / den).mean()


def combined_loss(logits, target):
    return F.binary_cross_entropy_with_logits(logits, target) + dice_loss(logits, target)


@torch.no_grad()
def dice_score(logits, target, thr=0.5, eps=1e-6):
    pred = (torch.sigmoid(logits) > thr).float()
    num = 2 * (pred * target).sum(dim=(2, 3))
    den = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + eps
    return ((num + eps) / den).mean().item()


# ────────────────────────────────────────────────────────────── 학습
def train(args):
    ds = DicomSegDataset(args.img_dir, args.mask_dir, augment=True)
    n_val = max(1, int(len(ds) * 0.2))
    idx = list(range(len(ds))); random.Random(42).shuffle(idx)
    val_idx, tr_idx = set(idx[:n_val]), set(idx[n_val:])
    tr = torch.utils.data.Subset(ds, sorted(tr_idx))
    va_base = DicomSegDataset(args.img_dir, args.mask_dir, augment=False)  # val은 증강 없음
    va = torch.utils.data.Subset(va_base, sorted(val_idx))
    print(f"train={len(tr)}  val={len(va)}  device={DEVICE}")

    tl = DataLoader(tr, batch_size=args.batch, shuffle=True, num_workers=0, drop_last=False)
    vl = DataLoader(va, batch_size=args.batch, shuffle=False, num_workers=0)

    model = UNet().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)
    scaler = torch.amp.GradScaler(DEVICE, enabled=(DEVICE == "cuda"))

    os.makedirs(args.out_dir, exist_ok=True)
    best = -1.0
    for ep in range(1, args.epochs + 1):
        model.train(); tr_loss = 0.0
        for img, msk in tl:
            img, msk = img.to(DEVICE), msk.to(DEVICE)
            opt.zero_grad()
            with torch.amp.autocast(DEVICE, enabled=(DEVICE == "cuda")):
                loss = combined_loss(model(img), msk)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tr_loss += loss.item() * img.size(0)
        tr_loss /= len(tr)
        sched.step()

        model.eval(); ds_sum = 0.0
        with torch.no_grad():
            for img, msk in vl:
                img, msk = img.to(DEVICE), msk.to(DEVICE)
                ds_sum += dice_score(model(img), msk) * img.size(0)
        val_dice = ds_sum / len(va)
        print(f"ep {ep:3d}/{args.epochs}  loss {tr_loss:.4f}  val_dice {val_dice:.4f}"
              + ("  *best" if val_dice > best else ""))
        if val_dice > best:
            best = val_dice
            torch.save({"model": model.state_dict(), "val_dice": best, "img_size": IMG_SIZE},
                       os.path.join(args.out_dir, "best.pt"))
    print(f"\n최고 val Dice = {best:.4f}  →  {os.path.join(args.out_dir, 'best.pt')}")


# ────────────────────────────────────────────────────────────── 추론
@torch.no_grad()
def predict(args):
    model = UNet().to(DEVICE)
    ckpt = torch.load(args.weights, map_location=DEVICE)
    model.load_state_dict(ckpt["model"]); model.eval()
    os.makedirs(args.out_dir, exist_ok=True)
    dcms = sorted(glob.glob(os.path.join(args.img_dir, "*.dcm")))
    print(f"{len(dcms)}장 추론 → {args.out_dir}")
    for dp in dcms:
        stem = os.path.splitext(os.path.basename(dp))[0]
        img = load_dicom(dp)
        x = torch.from_numpy(img)[None, None].to(DEVICE)
        prob = torch.sigmoid(model(x))[0, 0].cpu().numpy()
        mask = (prob > 0.5).astype(np.uint8)
        # 오버레이(초록) 저장
        rgb = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_GRAY2BGR)
        ov = rgb.copy(); ov[mask > 0] = (0, 200, 0)
        vis = cv2.addWeighted(ov, 0.4, rgb, 0.6, 0)
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(vis, cnts, -1, (0, 255, 0), 2)
        cv2.imwrite(os.path.join(args.out_dir, stem + "_overlay.png"), vis)
        cv2.imwrite(os.path.join(args.out_dir, stem + "_mask.png"), mask * 255)
    print("완료.")


# ────────────────────────────────────────────────────────────── 스모크 테스트
def smoke(_):
    """실데이터 없이 모델 forward / 손실 / Dice가 도는지 확인."""
    print(f"device={DEVICE}")
    m = UNet().to(DEVICE)
    x = torch.rand(2, 1, IMG_SIZE, IMG_SIZE, device=DEVICE)
    y = (torch.rand(2, 1, IMG_SIZE, IMG_SIZE, device=DEVICE) > 0.7).float()
    out = m(x)
    print("출력 shape:", tuple(out.shape), "(입력과 같아야 함)")
    print("combined_loss:", round(combined_loss(out, y).item(), 4))
    print("dice_score   :", round(dice_score(out, y), 4))
    n = sum(p.numel() for p in m.parameters())
    print(f"파라미터 수: {n/1e6:.1f}M")
    print("OK")


In [3]:
# ────────────────────────────────────────────────────────────── 설정
# 노트북에서는 argparse(명령줄 인자) 대신 설정 객체로 함수를 직접 호출한다.
class Cfg:
    img_dir  = "dataset-dcm/test/gather"
    mask_dir = "dataset-dcm/test/gather_mask/SegmentationClass"
    out_dir  = "unet_out"
    epochs   = 40
    batch    = 4
    lr       = 1e-3
    weights  = "unet_out/best.pt"

cfg = Cfg()
print("설정 준비 완료. 아래 셀에서 smoke → train → predict 순서로 실행하세요.")

설정 준비 완료. 아래 셀에서 smoke → train → predict 순서로 실행하세요.


In [4]:
# 1) 스모크 테스트 — 실데이터 없이 모델/손실/Dice가 도는지 확인
smoke(None)

device=cuda
출력 shape: (2, 1, 512, 512) (입력과 같아야 함)
combined_loss: 1.3358
dice_score   : 0.3857
파라미터 수: 7.8M
OK


In [5]:
# 2) 학습 — cfg의 설정으로 U-Net 학습. best.pt가 cfg.out_dir에 저장됨
train(cfg)

train=34  val=8  device=cuda
ep   1/40  loss 1.5895  val_dice 0.0000  *best
ep   2/40  loss 1.4149  val_dice 0.0000
ep   3/40  loss 1.3479  val_dice 0.0000
ep   4/40  loss 1.3204  val_dice 0.0000
ep   5/40  loss 1.2834  val_dice 0.0000
ep   6/40  loss 1.2528  val_dice 0.0000
ep   7/40  loss 1.2283  val_dice 0.0000
ep   8/40  loss 1.2080  val_dice 0.0000
ep   9/40  loss 1.1900  val_dice 0.0000
ep  10/40  loss 1.1740  val_dice 0.0000
ep  11/40  loss 1.1632  val_dice 0.0000
ep  12/40  loss 1.1528  val_dice 0.0000
ep  13/40  loss 1.1393  val_dice 0.0000
ep  14/40  loss 1.1288  val_dice 0.0000
ep  15/40  loss 1.1187  val_dice 0.0000
ep  16/40  loss 1.1115  val_dice 0.0000
ep  17/40  loss 1.1003  val_dice 0.0000
ep  18/40  loss 1.1032  val_dice 0.0000
ep  19/40  loss 1.0940  val_dice 0.0000
ep  20/40  loss 1.0904  val_dice 0.0000
ep  21/40  loss 1.0881  val_dice 0.0000
ep  22/40  loss 1.0800  val_dice 0.0000
ep  23/40  loss 1.0756  val_dice 0.0000
ep  24/40  loss 1.0668  val_dice 0.0000
ep  

In [6]:
# 3) 추론 — 학습된 best.pt로 전체 DICOM에 마스크/오버레이 저장
cfg.out_dir = "unet_out/pred"   # 추론 결과는 별도 폴더로
predict(cfg)

42장 추론 → unet_out/pred
완료.
